# Lab 1 – PCA · Dataset 2: Wine Quality (UCI ID 186)

Files: `winequality-red.csv`, `winequality-white.csv`. Separator is `;`.

In [195]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## I. Study the dataset
Each file is loaded separately. Cleaning is done per file.

In [196]:
DATA_DIR = "data"
red = pd.read_csv(f"{DATA_DIR}/winequality-red.csv", sep=";")
white = pd.read_csv(f"{DATA_DIR}/winequality-white.csv", sep=";")

datasets = {"red": red, "white": white}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

# Sanity check: 11 features + quality = 12 columns.
assert red.shape[1] == 12 and white.shape[1] == 12, "Wrong separator? Use sep=';'"
assert list(red.columns) == list(white.columns), "Column names differ between files"

red: 1599 rows, 12 columns
white: 4898 rows, 12 columns


In [197]:
red.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [198]:
# Column types: all should be numeric (float64, quality int64)
pd.DataFrame({"red": red.dtypes, "white": white.dtypes})

,red,white
fixed acidity,float64,float64
volatile acidity,float64,float64
citric acid,float64,float64
residual sugar,float64,float64
chlorides,float64,float64
free sulfur dioxide,float64,float64
total sulfur dioxide,float64,float64
density,float64,float64
pH,float64,float64
sulphates,float64,float64


### 1. Feature types

In [199]:
both = pd.concat([red, white])
feature_info = pd.DataFrame({
    "dtype": both.dtypes.astype(str),
    "n_unique": both.nunique(),
    "min": both.min(),
    "max": both.max(),
    "all_whole_numbers": (both % 1 == 0).all(),
})
feature_info

,dtype,n_unique,min,max,all_whole_numbers
fixed acidity,float64,106,3.80000,15.90000,False
volatile acidity,float64,187,0.08000,1.58000,False
citric acid,float64,89,0.00000,1.66000,False
residual sugar,float64,316,0.60000,65.80000,False
chlorides,float64,214,0.00900,0.61100,False
free sulfur dioxide,float64,135,1.00000,289.00000,False
total sulfur dioxide,float64,276,6.00000,440.00000,False
density,float64,998,0.98711,1.03898,False
pH,float64,108,2.72000,4.01000,False
sulphates,float64,111,0.22000,2.00000,False


Classification (based on the UCI dataset page and Cortez et al. 2009):

| Feature | Quantitative / Qualitative | Numerical / Categorical | Discrete / Continuous |
|---|---|---|---|
| `fixed acidity`, `volatile acidity`, `citric acid`, `residual sugar`, `chlorides`, `free sulfur dioxide`, `total sulfur dioxide`, `density`, `pH`, `sulphates`, `alcohol` | Quantitative | Numerical | Continuous |
| `quality` (label, score 0–10) | Qualitative (ordinal) | Categorical | Discrete |
| `type` (added at integration: red / white) | Qualitative (nominal) | Categorical | Discrete |

Note: `quality` is the median of expert grades on an ordered scale: the order matters, but the distance between two grades is not a measured quantity.

### 2. Missing values

In [200]:
missing = pd.DataFrame({
    "red_missing": red.isna().sum(),
    "white_missing": white.isna().sum(),
})
print("Rows with at least one missing value:")
for name, df in datasets.items():
    print(f"  {name}: {df.isna().any(axis=1).sum()}")
missing

Rows with at least one missing value:
  red: 0
  white: 0


,red_missing,white_missing
fixed acidity,0,0
volatile acidity,0,0
citric acid,0,0
residual sugar,0,0
chlorides,0,0
free sulfur dioxide,0,0
total sulfur dioxide,0,0
density,0,0
pH,0,0
sulphates,0,0


### 3. Duplicate records

In [201]:
for name, df in datasets.items():
    n_dup = df.duplicated().sum()
    print(f"{name}: {n_dup} duplicate rows ({n_dup / len(df) * 100:.2f}%)")

red: 240 duplicate rows (15.01%)
white: 937 duplicate rows (19.13%)


In [202]:
# Look at some duplicated groups (keep=False shows all copies)
red[red.duplicated(keep=False)].sort_values(list(red.columns)).head(6)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
142,5.2,0.34,0.00,1.8,0.050,27.0,63.0,0.9916,3.68,0.79,14.0,6
144,5.2,0.34,0.00,1.8,0.050,27.0,63.0,0.9916,3.68,0.79,14.0,6
131,5.6,0.50,0.09,2.3,0.049,17.0,99.0,0.9937,3.63,0.63,13.0,5
132,5.6,0.50,0.09,2.3,0.049,17.0,99.0,0.9937,3.63,0.63,13.0,5
1488,5.6,0.54,0.04,1.7,0.049,5.0,13.0,0.9942,3.72,0.58,11.4,5
1491,5.6,0.54,0.04,1.7,0.049,5.0,13.0,0.9942,3.72,0.58,11.4,5


### 4. Inconsistency
Rows with the same 11 features but a different `quality`

In [203]:
features = [c for c in red.columns if c != "quality"]

for name, df in datasets.items():
    same_features = df.duplicated(subset=features, keep=False)
    groups = df[same_features].groupby(features)["quality"].nunique()
    n_conflict = (groups > 1).sum()
    print(f"{name}: {(groups > 0).sum()} duplicate feature groups, "
          f"{n_conflict} with different quality")

red: 220 duplicate feature groups, 0 with different quality
white: 772 duplicate feature groups, 0 with different quality


### Drop duplicates

In [204]:
red_clean = red.drop_duplicates().reset_index(drop=True)
white_clean = white.drop_duplicates().reset_index(drop=True)
datasets_clean = {"red": red_clean, "white": white_clean}

for name in datasets:
    before, after = len(datasets[name]), len(datasets_clean[name])
    print(f"{name}: {before} -> {after} rows ({before - after} removed, "
          f"{(before - after) / before * 100:.2f}%)")

red: 1599 -> 1359 rows (240 removed, 15.01%)
white: 4898 -> 3961 rows (937 removed, 19.13%)


### 5. Incorrect values

In [205]:
# --- Incorrect values (checked on duplicate-free data) ---
features = [c for c in red_clean.columns if c != 'quality']

def check_incorrect(df, name):
    rules = {
        'Negative value (any feature)': (df[features] < 0).any(axis=1),
        'pH outside 0-14': ~df['pH'].between(0, 14),
        'Alcohol outside 0-100 % vol': ~df['alcohol'].between(0, 100),
        'Density <= 0': df['density'] <= 0,
        'Free SO2 > total SO2': df['free sulfur dioxide'] > df['total sulfur dioxide'],
        'Quality not an integer 0-10': ~df['quality'].between(0, 10) | (df['quality'] % 1 != 0),
    }
    result = pd.DataFrame({'Rows flagged': {k: int(v.sum()) for k, v in rules.items()}})
    result['Percentage (%)'] = (result['Rows flagged'] / len(df) * 100).round(2)
    any_flag = pd.concat(rules.values(), axis=1).any(axis=1)
    print(f'{name}: {int(any_flag.sum())} of {len(df)} rows have at least one incorrect value')
    return result, any_flag

res_red, bad_red = check_incorrect(red_clean, 'Red')
res_white, bad_white = check_incorrect(white_clean, 'White')
display(pd.concat({'Red': res_red, 'White': res_white}, axis=1))

Red: 0 of 1359 rows have at least one incorrect value
White: 0 of 3961 rows have at least one incorrect value


Red                       White  \
                             Rows flagged Percentage (%) Rows flagged   
Negative value (any feature)            0            0.0            0   
pH outside 0-14                         0            0.0            0   
Alcohol outside 0-100 % vol             0            0.0            0   
Density <= 0                            0            0.0            0   
Free SO2 > total SO2                    0            0.0            0   
Quality not an integer 0-10             0            0.0            0   

                                             
                             Percentage (%)  
Negative value (any feature)            0.0  
pH outside 0-14                         0.0  
Alcohol outside 0-100 % vol             0.0  
Density <= 0                            0.0  
Free SO2 > total SO2                    0.0  
Quality not an integer 0-10             0.0

In [206]:
display(pd.concat({'Red': red_clean[features].agg(['min', 'max']).T,
                   'White': white_clean[features].agg(['min', 'max']).T}, axis=1))

Red               White           
                          min        max      min        max
fixed acidity         4.60000   15.90000  3.80000   14.20000
volatile acidity      0.12000    1.58000  0.08000    1.10000
citric acid           0.00000    1.00000  0.00000    1.66000
residual sugar        0.90000   15.50000  0.60000   65.80000
chlorides             0.01200    0.61100  0.00900    0.34600
free sulfur dioxide   1.00000   72.00000  2.00000  289.00000
total sulfur dioxide  6.00000  289.00000  9.00000  440.00000
density               0.99007    1.00369  0.98711    1.03898
pH                    2.74000    4.01000  2.72000    3.82000
sulphates             0.33000    2.00000  0.22000    1.08000
alcohol               8.40000   14.90000  8.00000   14.20000

### 6. Outliers (IQR)

In [207]:
# --- Outliers (IQR, 1.5 x IQR), on duplicate-free data ---
def iqr_outliers(df, name):
    X = df[features]
    q1, q3 = X.quantile(0.25), X.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flags = (X < lower) | (X > upper)
    table = pd.DataFrame({
        'Lower bound': lower.round(4),
        'Upper bound': upper.round(4),
        'Flagged outliers': flags.sum(),
        'Percentage (%)': (flags.sum() / len(df) * 100).round(2),
    })
    any_out = flags.any(axis=1)
    print(f'{name}: {int(any_out.sum())} of {len(df)} rows '
          f'({any_out.mean() * 100:.2f}%) have at least one outlier')
    return table

out_red = iqr_outliers(red_clean, 'Red')
out_white = iqr_outliers(white_clean, 'White')
display(pd.concat({'Red': out_red, 'White': out_white}, axis=1))

Red: 340 of 1359 rows (25.02%) have at least one outlier
White: 706 of 3961 rows (17.82%) have at least one outlier


Red                                              \
                     Lower bound Upper bound Flagged outliers Percentage (%)   
fixed acidity             3.9500     12.3500               41           3.02   
volatile acidity          0.0150      1.0150               19           1.40   
citric acid              -0.4200      0.9400                1           0.07   
residual sugar            0.8500      3.6500              126           9.27   
chlorides                 0.0385      0.1225               87           6.40   
free sulfur dioxide     -14.0000     42.0000               26           1.91   
total sulfur dioxide    -39.5000    124.5000               45           3.31   
density                   0.9923      1.0012               35           2.58   
pH                        2.9250      3.6850               28           2.06   
sulphates                 0.2800      1.0000               55           4.05   
alcohol                   7.1000     13.5000               12           0.88   

                           White                                              
                     Lower bound Upper bound Flagged outliers Percentage (%)  
fixed acidity             4.8000      8.8000              106           2.68  
volatile acidity          0.0300      0.5100              133           3.36  
citric acid               0.0900      0.5700              223           5.63  
residual sugar           -9.3500     19.8500               16           0.40  
chlorides                 0.0125      0.0725              178           4.49  
free sulfur dioxide     -10.0000     78.0000               44           1.11  
total sulfur dioxide     16.0000    256.0000               14           0.35  
density                   0.9855      1.0018                6           0.15  
pH                        2.7900      3.5900               46           1.16  
sulphates                 0.2000      0.7600               96           2.42  
alcohol                   6.6500     14.2500                0           0.00

## Cleaning summary (before/after)

In [208]:
# --- Cleaning summary: before (raw) vs after (clean), per file ---
red_raw, white_raw = red, white

def n_inconsistent(df):
    n_q = df.groupby(features)['quality'].transform('nunique')
    return int((n_q > 1).sum())

def n_out_rows(df):
    X = df[features]
    q1, q3 = X.quantile(0.25), X.quantile(0.75)
    iqr = q3 - q1
    return int(((X < q1 - 1.5 * iqr) | (X > q3 + 1.5 * iqr)).any(axis=1).sum())

def summary(raw, clean, bad_clean):
    return pd.DataFrame({
        'Before': [len(raw), int(raw.isna().any(axis=1).sum()), int(raw.duplicated().sum()),
                   n_inconsistent(raw), int(bad_clean.sum()), '-'],
        'After':  [len(clean), int(clean.isna().any(axis=1).sum()), int(clean.duplicated().sum()),
                   n_inconsistent(clean), int(bad_clean.sum()), f'{n_out_rows(clean)} (kept)'],
    }, index=['Total records', 'Records with missing values', 'Duplicate records',
              'Records with detected inconsistencies', 'Records with incorrect values',
              'Records with at least one IQR outlier'])

tab = pd.concat({'Red': summary(red_raw, red_clean, bad_red),
                 'White': summary(white_raw, white_clean, bad_white)}, axis=1)
display(tab)

for name, raw in [('Red', red_raw), ('White', white_raw)]:
    print(f'{name}: {raw.duplicated().sum() / len(raw) * 100:.2f}% duplicates')
    
print('\nRemoved records - Red:', len(red_raw) - len(red_clean),
      '| White:', len(white_raw) - len(white_clean))

Red              White            
                                      Before       After Before       After
Total records                           1599        1359   4898        3961
Records with missing values                0           0      0           0
Duplicate records                        240           0    937           0
Records with detected inconsistencies      0           0      0           0
Records with incorrect values              0           0      0           0
Records with at least one IQR outlier      -  340 (kept)      -  706 (kept)

Red: 15.01% duplicates
White: 19.13% duplicates

Removed records - Red: 240 | White: 937


## Data integration

In [209]:
# Schema check: both files must have the same columns, in the same order
print("Same columns:", list(red_clean.columns) == list(white_clean.columns))

features = list(red_clean.columns.drop('quality'))   # the 11 physicochemical features

# Add the type column and stack the two files (vertical merge, not a join)
wine = pd.concat(
    [red_clean.assign(type='red'), white_clean.assign(type='white')],
    ignore_index=True
)
print(wine.shape)
wine.head()

Same columns: True
(5320, 13)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5,red


In [210]:
# Rows that appear in both files (same 11 features + quality)
common = red_clean.merge(white_clean, on=list(red_clean.columns), how='inner')

integ_tab = pd.DataFrame({
    'Result': [
        len(red_clean),
        len(white_clean),
        len(wine),
        len(wine) == len(red_clean) + len(white_clean),
        int(wine.isna().sum().sum()),
        len(common),
    ]
}, index=[
    'Rows red (after cleaning)',
    'Rows white (after cleaning)',
    'Rows after integration',
    'No rows lost or duplicated',
    'Missing values after integration',
    'Identical rows found in both files',
])
integ_tab

,Result
Rows red (after cleaning),1359
Rows white (after cleaning),3961
Rows after integration,5320
No rows lost or duplicated,True
Missing values after integration,0
Identical rows found in both files,2


In [211]:
type_tab = wine['type'].value_counts().to_frame('Count')
type_tab['Percentage (%)'] = (type_tab['Count'] / len(wine) * 100).round(2)
type_tab

,Count,Percentage (%)
type,,
white,3961,74.45
red,1359,25.55


In [212]:
wine.groupby('type')[features].mean().T.round(3)

type,red,white
fixed acidity,8.311,6.839
volatile acidity,0.529,0.281
citric acid,0.272,0.334
residual sugar,2.523,5.915
chlorides,0.088,0.046
free sulfur dioxide,15.893,34.889
total sulfur dioxide,46.826,137.194
density,0.997,0.994
pH,3.310,3.195
sulphates,0.659,0.490
